# Make summary plots of the titers of different strains against human sera

## Setup and read data

In [ ]:
import itertools

import altair as alt

import pandas as pd

_ = alt.data_transformers.disable_max_rows()

Get variables from `snakemake`:

In [ ]:
metadata_csv = snakemake.input.metadata_csv
titers_csv = snakemake.input.titers_csv
virus_csv = snakemake.input.virus_csv
viral_strain_plot_order_csv = snakemake.input.viral_strain_plot_order
recent_vaccine_strains = snakemake.params.recent_vaccine_strains
human_sera_plots_params = snakemake.params.human_sera_plots_params
summarized_titers_csv = snakemake.output.summarized_titers_csv
sera_collection_date_and_age_plot = snakemake.output.sera_collection_date_and_age_plot
chart_htmls = snakemake.output.chart_htmls

In [ ]:
metadata_all = pd.read_csv(metadata_csv)
print(f"Read {len(metadata_all)=} for sera from {metadata_csv=}")

titers_all = pd.read_csv(titers_csv)
print(f"\nRead {len(titers_all)=} titers from {titers_csv=}")
assert set(titers_all["serum"]) == set(metadata_all["serum"])

viruses_all = (
    pd.read_csv(virus_csv)
    [["strain", "subtype", "strain_type", "subclade"]]
    .drop_duplicates()
    .rename(columns={"strain": "virus"})
)
assert set(titers_all["virus"]).issubset(viruses_all["virus"])
assert set(recent_vaccine_strains).issubset(viruses_all["virus"])
viruses_all["strain_type"] = viruses_all["strain_type"].where(
    ~viruses_all["virus"].isin(recent_vaccine_strains), "recent_vaccine"
)
print(f"\nRead {len(viruses_all)=} viruses from {virus_csv=}")

viral_strain_plot_order = pd.read_csv(viral_strain_plot_order_csv)["strain"].tolist()
assert set(viruses_all["virus"]).issubset(viral_strain_plot_order)

## Get fraction of sera and viruses with titers
Look at the fraction of sera and viruses with titers.
We generally may want to drop sera that lack titers for many viruses, and viruses that lack titers for many sera.
Depending on `min_frac_action` (specified in configuration), we either raise an error if any sera or viruses are below the fractions specified in the configuration, or drop any sera or titers below these fractions.
In general for production runs you may want to raise an error and filter these at an upstream step.

In [ ]:
min_frac_action = human_sera_plots_params["min_frac_action"]

titers = titers_all.copy()
metadata = metadata_all.copy()
viruses = viruses_all.copy()

for min_frac_type, vals, frac_vals, col, frac_var in [
    ("min_frac_strains", set(metadata["serum"]), set(viruses["virus"]), "serum", "virus"),
    ("min_frac_sera", set(viruses["virus"]), set(metadata["serum"]), "virus", "serum"),
]:
    min_frac = human_sera_plots_params[min_frac_type]
    print(f"\nChecking for {min_frac_type=} at cutoff of {min_frac=}")
    frac_df = (
        titers
        .groupby(col, as_index=False)
        .aggregate(n_w_titers=pd.NamedAgg(col, "count"))
        .assign(frac_w_titers=lambda x: x["n_w_titers"] / len(frac_vals))
    )
    frac_df = pd.concat(
        [
            frac_df,
            pd.DataFrame(
                {
                    col: [v for v in vals if v not in set(frac_df[col])],
                    "n_w_titers": 0,
                    "frac_w_titers": 0.0,
                }
            )
        ],
        ignore_index=True,
    )
    assert len(frac_df) == len(vals)
    assert (frac_df["frac_w_titers"] <= 1).all()

    frac_df["below_cutoff"] = frac_df["frac_w_titers"] < min_frac

    frac_chart = (
        alt.Chart(frac_df)
        .encode(
            alt.X(
                "frac_w_titers",
                title=f"fraction {frac_var} with titers",
                scale=alt.Scale(domain=[0, 1]),
            ),
            alt.Y(col, sort=alt.SortField("frac_w_titers", order="descending")),
            alt.Fill("below_cutoff", title=f"below cutoff of {min_frac_type} = {min_frac}"),
            tooltip=[col, "n_w_titers", alt.Tooltip("frac_w_titers", format=".3f")],
        )
        .mark_bar()
        .properties(
            height=alt.Step(11),
            width=135,
            title=f"{frac_var} with titers for each {col}",
        )
        .configure_axis(labelLimit=500)
        .configure_legend(titleLimit=500)
    )

    display(frac_chart)
    
    failed = frac_df.query("below_cutoff").sort_values("frac_w_titers").reset_index(drop=True)
    below_frac = set(failed[col])
    print(f"Overall, {len(below_frac)} {col} have titers for less than {min_frac} {frac_var}")
    display(failed)

    if min_frac_action == "raise":
        if not failed.empty:
            raise ValueError(frac_df.query("below_cutoff").sort_values("frac_w_titers").reset_index(drop=True))
    elif min_frac_action == "drop":
        if not failed.empty:
            print(f"Dropping these {col}")
            titers = titers[~titers[col].isin(below_frac)]
            if col == "serum":
                metadata = metadata[~metadata[col].isin(below_frac)]
            if col == "virus":
                viruses = viruses[~viruses[col].isin(below_frac)]
    else:
        raise ValueError(f"invalid {min_frac_action=}")

print(
    f"\nAfter any filtering:"
    f"\n {len(titers)=} / {len(titers_all)=}"
    f"\n {len(viruses)=} / {len(viruses_all)=}"
    f"\n {len(metadata)=} / {len(metadata_all)=}"
)

## Plot age and collection distributions of sera

In [ ]:
sera_base = (
    alt.Chart(
        metadata[["group", "serum", "collection_date_numerical", "age_years"]]
        .assign(
            n=lambda x: x.groupby("group")["serum"].transform("count"),
            group=lambda x: x["group"] + " (n=" + x["n"].astype(str) + ")",
        )
        .drop(columns="n")
    )
    .encode(alt.Row("group", title=None))
    .mark_bar()
)

sera_date_chart = (
    sera_base
    .encode(
        alt.X(
            "yearmonth(collection_date_numerical)",
            title="collection date",
            axis=alt.Axis(format="%b-%Y", labelAngle=270),
            scale=alt.Scale(nice=False, padding=3),
        ),
        alt.Y("count()", title="number of sera", scale=alt.Scale(nice=False, padding=3)),
    )
    .resolve_scale(y="independent")
    .properties(height=80, width=70)
)

sera_age_chart = (
    sera_base
    .encode(
        alt.X(
            "age_years",
            title="age (years)",
            bin=alt.Bin(step=5, anchor=0),
            scale=alt.Scale(nice=False, padding=3),
        ),
        alt.Y("count()", title="number of sera", scale=alt.Scale(nice=False, padding=3)),
    )
    .resolve_scale(y="independent")
    .properties(height=80, width=150)
)

sera_chart = (
    alt.hconcat(sera_date_chart, sera_age_chart)
    .configure_axis(grid=False, titleFontWeight="normal")
    .configure_header(
        title=None, labelOrient="top", labelFontSize=11, labelPadding=2
    )
    .configure_facet(spacing=7)
    .configure_view(stroke="black")
    .properties(
        title=alt.TitleParams(
            "collection dates and subject ages for sera",
            anchor="middle",
        ),
    )
)

print(f"Saving to {sera_collection_date_and_age_plot}")
sera_chart.save(sera_collection_date_and_age_plot)

sera_chart

## Plot all the titers

### Chart base and selections

In [ ]:
assert len(titers) == len(titers.groupby(["serum", "virus"]))
assert viruses["virus"].nunique() == len(viruses.groupby(["virus", "subtype", "strain_type"]))

# for group labels within altair we calculate these to have n too
groups = (
    pd.concat([metadata, metadata.assign(group="All")])
    .groupby("group", as_index=False)
    .aggregate(n=pd.NamedAgg("serum", "nunique"))
    .assign(group=lambda x: x["group"] + " (n=" + x["n"].astype(str) + ")")
    ["group"].tolist()
)
groups = ["All", *sorted(metadata["group"].unique())]
print(f"{groups=}")

virus_selection = alt.selection_point(
    fields=["virus"], on="mouseover", empty=False, clear="mouseout", nearest=False
)

serum_selection = alt.selection_point(
    fields=["serum"], on="mouseover", empty=False, clear="mouseout", nearest=False
)

group_selection = alt.selection_point(
    fields=["group"],
    bind=alt.binding_select(
        name="serum group", options=[None, *groups], labels=["show all", *groups]
    ),
    init=None,
)

max_age = 5 * int(metadata["age_years"].max() // 5) + 5
assert all(metadata["age_years"] <= max_age)
min_age_slider = alt.param(
    value=0,
    bind=alt.binding_range(min=0, max=max_age, step=5, name="minimum subject age (years)"),
)
max_age_slider = alt.param(
    value=max_age,
    bind=alt.binding_range(min=0, max=max_age, step=5, name="maximum subject age (years)"),
)

# make the chart base, using transform_lookup to make it as small as possible
# by looking up serum-specific and virus-specific annotations
titers_base_nolookup = (
    alt.Chart(titers[["serum", "virus", "titer"]])
    .add_params(
        virus_selection,
        serum_selection,
        group_selection,
        min_age_slider,
        max_age_slider,
    )
    .encode(
        alt.Y(
            "virus",
            sort=viral_strain_plot_order,
            axis=alt.Axis(
                labelLimit=500,
                labelExpr="replace(datum.label, regexp('_[^_]*$'), '')",  # remove _H1N1 or _H3N2
            ),
        ),
    )
    .properties(height=alt.Step(11), width=135)
)

# because of scoping issues when layering and faceting charts with
# transform_lookups (faceting must be done before lookups), we add
# this function to do the faceting and lookups
def facet_and_add_lookups(chart):
    return (
        chart
        # facet
        .facet(column=alt.Column("group_n:N", title=None))
        # lookup additional data
        .transform_lookup(
            lookup="serum",
            from_=alt.LookupData(
                data=metadata,
                key="serum",
                fields=["group", "collection_date_string", "age_string", "age_years", "sex"],
            ),
        )
        .transform_lookup(
            lookup="virus",
            from_=alt.LookupData(
                data=viruses,
                key="virus",
                fields=["subtype", "strain_type", "subclade"],
            ),
        )
        # trick to make a new variable with all groups
        .transform_calculate(facet_with_all="[datum.group, 'All']")
        .transform_flatten(["facet_with_all"], as_=["group"])
        # filter by group and age
        .transform_filter(group_selection)
        .transform_filter(alt.datum["age_years"] >= min_age_slider)
        .transform_filter(alt.datum["age_years"] <= max_age_slider)
        # make facet labels w n per group
        .transform_joinaggregate(n_per_group="distinct(serum)", groupby=["group"])
        .transform_calculate(group_n="datum.group + ' (n=' + datum.n_per_group + ')'")
    )

In [ ]:
# set titer scale
titer_lower_limit = human_sera_plots_params["titer_lower_limit"]
print(f"Using {titer_lower_limit=}")
titer_scale = alt.Scale(type="log", nice=False, domainMin=titer_lower_limit, padding=4)

### Median titers chart

In [ ]:
# make median titer point chart
median_points = (
    titers_base_nolookup
    .transform_aggregate(
        median_titer="median(titer)",
        groupby=["virus", "subtype", "strain_type", "subclade", "group"],
    )
    .encode(
        alt.X("median_titer:Q", title="titer", scale=titer_scale),
        tooltip=["virus", alt.Tooltip("median_titer:Q", format=".1f"), "strain_type:N", "subclade:N"],
        color=alt.condition(virus_selection, alt.value("red"), alt.value("black")),
        size=alt.condition(virus_selection, alt.value(80), alt.value(40)),
    )
    .mark_circle(opacity=1)
)

#facet_and_add_lookups(median_points)

### Per-serum line charts

In [ ]:
# make per-serum lines
serum_lines =  (
    titers_base_nolookup
    .encode(
        alt.X("titer", scale=titer_scale),
        alt.Detail("serum"),
        tooltip=[
            "virus",
            "serum",
            alt.Tooltip("titer", format=".1f"),
            alt.Tooltip("collection_date_string:N", title="serum date"),
            alt.Tooltip("age_string:N", title="age"),
            "sex:N",
        ],
        size=alt.condition(serum_selection, alt.value(3), alt.value(1.5)),
        opacity=alt.condition(serum_selection, alt.value(1), alt.value(0.2)),
    )
    .mark_line()
)

#facet_and_add_lookups(serum_lines + median_points)

### Interquartile range chart

In [ ]:
interquartile_range =  (
    titers_base_nolookup
    .transform_joinaggregate(
        median_titer="median(titer)",
        titer_q1="q1(titer)",
        titer_q3="q3(titer)",
        groupby=["virus"],
    )
    .encode(
        alt.X("titer", scale=titer_scale),
        tooltip=[
            "virus",
            alt.Tooltip("median_titer:Q", format=".1f"),
            alt.Tooltip("titer_q1:Q", format=".1f"),
            alt.Tooltip("titer_q3:Q", format=".1f"),
            "strain_type:N",
            "subclade:N",
        ],
    )
    .mark_errorband(extent="iqr", opacity=0.5, interpolate="linear")
)

#facet_and_add_lookups(interquartile_range + median_points)

### Fraction below titer cutoff chart

In [ ]:
titer_cutoff = human_sera_plots_params["titer_cutoff"]
print(f"Setting initial {titer_cutoff=}")

titer_cutoff_slider = alt.param(
    value=titer_cutoff,
    bind=alt.binding_range(
        min=titer_lower_limit,
        max=1000,
        step=5,
        name="fraction sera below this cutoff",
    ),
)

# make titer cutoff chart
frac_below_cutoff = (
    titers_base_nolookup
    .add_params(titer_cutoff_slider)
    .transform_calculate(below_cutoff=alt.datum["titer"] < titer_cutoff_slider)
    .transform_aggregate(
        n_below_cutoff="sum(below_cutoff)",
        n_total="distinct(serum)",
        groupby=["virus", "subtype", "strain_type", "subclade", "group"],
    )
    .transform_calculate(
        frac_below_cutoff=alt.datum["n_below_cutoff"] / alt.datum["n_total"]
    )
    .encode(
        alt.X("frac_below_cutoff:Q", title="fraction below cutoff"),
        tooltip=["virus", alt.Tooltip("frac_below_cutoff:Q", format=".2f"), "strain_type:N", "subclade:N"],
        color=alt.condition(virus_selection, alt.value("red"), alt.value("black")),
    )
    .mark_bar(opacity=0.8)
)

#facet_and_add_lookups(frac_below_cutoff)

### Now make nicely formatted charts and save them

In [ ]:
made_chart = {c: False for c in chart_htmls}

for subtype, strain_type, (chart_obj, chart_desc, title) in itertools.product(
    viruses["subtype"].unique(),
    ["recent", "vaccine"],
    [
        ((serum_lines + median_points), "individual_sera", "median (points) and per-serum (lines) titers"),
        ((interquartile_range + median_points), "interquartile_range", "median (points) and interquartile range titers"),
        (frac_below_cutoff, "frac_below_cutoff", "fraction sera below titer cutoff"),
    ],
):
    filesuffix = f"{subtype}_{strain_type}_{chart_desc}.html"
    filename = [c for c in chart_htmls if filesuffix in c]
    assert len(filename) == 1, f"did not find one {filesuffix=} in {chart_htmls=}"
    filename = filename[0]
    made_chart[filename] = True

    chart = (
        facet_and_add_lookups(chart_obj)
        .transform_filter(alt.datum["subtype"] == subtype)
        .transform_filter(
            alt.FieldOneOfPredicate(
                "strain_type",
                {
                    "recent": ["circulating_2025", "recent_vaccine"],
                    "vaccine": ["vaccine", "recent_vaccine"],
                }[strain_type]
            )
        )
        .configure_axis(grid=False, titleFontWeight="normal", titleFontSize=13)
        .configure_header(
            title=None, labelOrient="top", labelFontSize=13, labelPadding=2
        )
        .configure_view(stroke="black")
        .configure_facet(spacing=8)
        .properties(
            title=alt.TitleParams(
                f"{title} for {subtype} {strain_type} strains",anchor="middle", fontSize=13
            ),
        )
    )
    display(chart)

    print(f"Saving to {filename=}\n")
    chart.save(filename)

assert all(made_chart.values()), f"{made_chart=}"

## Make data frame / CSV of summarized titers
The above plots summarize the per-virus titers with several statistics: medians, interquartile range, and fraction below cutoff.
Here we make and write a data frame with these summarized values.

In [ ]:
summarized_titers = (
    pd.concat([titers.assign(group="All"), titers])
    .merge(viruses, on=["virus"], how="left", validate="many_to_one")
    .groupby(["subtype", "strain_type", "subclade", "virus", "group"], as_index=False)
    .aggregate(
        median_titer=pd.NamedAgg("titer", "median"),
        titer_q1=pd.NamedAgg("titer", lambda s: s.quantile(0.25)),
        titer_q3=pd.NamedAgg("titer", lambda s: s.quantile(0.75)),
        frac_below_cutoff=pd.NamedAgg(
            "titer", lambda s: (s < titer_cutoff).sum() / len(s)
        ),
    )
    .rename(
        columns={
            "frac_below_cutoff": f"frac_w_titer_below_{titer_cutoff}",
            "group": "serum_group",
        }
    )
)

print(f"Saving summarized titers to {summarized_titers_csv=}")
summarized_titers.to_csv(summarized_titers_csv, float_format="%.3f", index=False)

summarized_titers